In [ ]:
# ── Config ──────────────────────────────────────────────────
import os; os.makedirs('output', exist_ok=True)

TUBULARITY_NPZ = 'output/tubularity_anchored.npz'
SOMA_JSON      = 'output/soma.json'
OUT_NPZ        = 'output/prep_riem.npz'
DOWNSAMPLE     = 4

In [ ]:
# ── Load + Downsample ───────────────────────────────────────
import numpy as np, json, gc, time
from scipy.ndimage import zoom

t0 = time.time()

tub          = np.load(TUBULARITY_NPZ)
T            = tub['T_combined'].astype(np.float32)
radius       = tub['radius_map'].astype(np.float32)
orient_field = tub['orient_field'].astype(np.float32)   # (Z,Y,X,3) float16→float32
soma_mask    = tub['soma_mask']
voxel_iso    = float(tub['voxel_iso'])
del tub; gc.collect()

with open(SOMA_JSON) as f:
    soma = json.load(f)
soma_vox  = np.array(soma['centroid_vox'], dtype=np.float32)
soma_r_um = float(soma['radius_um'])

factor = 1.0 / DOWNSAMPLE

T_down         = zoom(T,         factor, order=1).astype(np.float32)
radius_down    = zoom(radius,    factor, order=1).astype(np.float32)
soma_mask_down = zoom(soma_mask, factor, order=0).astype(bool)
del T, radius, soma_mask; gc.collect()

# orient_field 다운샘플: 각 채널 따로 zoom → 재정규화
print('Downsampling orient_field...', flush=True)
vz = zoom(orient_field[..., 0], factor, order=1).astype(np.float32)
vy = zoom(orient_field[..., 1], factor, order=1).astype(np.float32)
vx = zoom(orient_field[..., 2], factor, order=1).astype(np.float32)
del orient_field; gc.collect()

norm = np.sqrt(vz**2 + vy**2 + vx**2) + 1e-8
vz /= norm; vy /= norm; vx /= norm
orient_down = np.stack([vz, vy, vx], axis=-1).astype(np.float16)   # (Zd,Yd,Xd,3)
del vz, vy, vx, norm; gc.collect()

voxel_down    = voxel_iso * DOWNSAMPLE
soma_vox_down = (soma_vox * factor).astype(np.float32)
Zd, Yd, Xd   = T_down.shape

print(f'T_down         : {T_down.shape}  voxel={voxel_down:.3f} µm  ({time.time()-t0:.1f}s)')
print(f'orient_down    : {orient_down.shape}  dtype={orient_down.dtype}')
print(f'soma_mask_down : {soma_mask_down.sum():,} voxels')
print(f'T range        : {T_down.min():.4f} – {T_down.max():.4f}')
print(f'soma_vox_down  : {soma_vox_down}  r={soma_r_um:.2f} µm')

In [ ]:
# ── CHECK ────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# orient_field norm should be ≈1 everywhere (non-zero T)
of32   = orient_down.astype(np.float32)
v_norm = np.sqrt((of32**2).sum(axis=-1))
mask   = T_down > 0.05
print(f'orient norm (T>0.05): mean={v_norm[mask].mean():.4f}  std={v_norm[mask].std():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
axes[0].imshow(T_down.max(axis=0), cmap='hot', vmin=0, vmax=1)
axes[0].scatter([soma_vox_down[2]], [soma_vox_down[1]], c='cyan', s=60, marker='+', zorder=5)
axes[0].set_title('T_down XY MIP'); axes[0].axis('off')

# v_OOF direction: colour by vx (X component)
axes[1].imshow(of32[..., 2].max(axis=0), cmap='RdBu', vmin=-1, vmax=1)
axes[1].set_title('orient vx (X component) MIP'); axes[1].axis('off')

axes[2].imshow(v_norm.max(axis=0), cmap='viridis', vmin=0, vmax=1.1)
axes[2].set_title('orient norm MIP (should be ≈1)'); axes[2].axis('off')

plt.show()

In [ ]:
# ── EDT radius ───────────────────────────────────────────────
from scipy.ndimage import distance_transform_edt, gaussian_filter

EDT_THRESHOLD  = 0.08   # T_down foreground 기준
EDT_SMOOTH_SIG = 2.0    # Gaussian sigma (voxel) — aliasing 제거

t0 = time.time()
fg_mask = T_down > EDT_THRESHOLD

# foreground(non-zero) → distance to nearest background(zero) = local tube radius
edt_raw  = distance_transform_edt(fg_mask).astype(np.float32) * voxel_down
edt_down = gaussian_filter(edt_raw, sigma=EDT_SMOOTH_SIG).astype(np.float32)
del edt_raw; gc.collect()

fg_edt = edt_down[fg_mask]
print(f'EDT done in {time.time()-t0:.1f}s')
print(f'edt (foreground only):')
print(f'  min={fg_edt.min():.3f}  median={np.median(fg_edt):.3f}'
      f'  p99={np.percentile(fg_edt,99):.3f}  max={fg_edt.max():.3f} µm')
del fg_edt

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
axes[0].imshow(edt_down.max(axis=0), cmap='hot')
axes[0].set_title('EDT radius MIP  (brighter=thicker)'); axes[0].axis('off')
axes[1].hist(edt_down[fg_mask].ravel(), bins=60, color='steelblue')
axes[1].set_xlabel('EDT radius (um)'); axes[1].set_title('EDT histogram (foreground only)')
plt.show()

In [ ]:
# ── Save ────────────────────────────────────────────────────
np.savez_compressed(OUT_NPZ,
    T_down         = T_down,
    radius_down    = radius_down,
    edt_down       = edt_down,
    orient_down    = orient_down,
    soma_mask_down = soma_mask_down,
    voxel_down     = np.float32(voxel_down),
    voxel_iso      = np.float32(voxel_iso),
    downsample     = np.int32(DOWNSAMPLE),
    soma_vox       = soma_vox,
    soma_vox_down  = soma_vox_down,
    soma_r_um      = np.float32(soma_r_um),
)
print(f'Saved: {OUT_NPZ}')
print(f'  T_down         {T_down.shape}  float32')
print(f'  edt_down       {edt_down.shape}  float32  ← EDT radius')
print(f'  radius_down    {radius_down.shape}  float32  ← OOF radius (보존)')
print(f'  orient_down    {orient_down.shape}  float16')
print(f'  soma_mask_down {soma_mask_down.shape}  bool')
print(f'  voxel_down     {voxel_down:.3f} µm')